<div style="text-align: center;">
  <h1>MCP — Model Context Protocol</h1>
  <h3>Máster en Data Science e IA · EBIS Business Techschool</h3>
  <p>Clase 3: Agentes y Vibe Coding</p>
</div>

**MCP** es el estándar abierto para conectar un agente con herramientas y datos: **escribes el servidor una vez y lo usa
cualquier cliente** (Claude, ChatGPT, VS Code, Cursor, el Agents SDK, LangGraph…). Nació en Anthropic (nov. 2024), hoy lo
gobierna la Agentic AI Foundation (Linux Foundation) y la especificación de julio de 2026 lo hace **sin estado**, más fácil de escalar.

Un servidor MCP ofrece tres tipos de cosas:

| Primitiva | Qué es | Quién decide usarla |
|---|---|---|
| **Tools** | funciones que el agente puede ejecutar | el modelo |
| **Resources** | datos que el agente puede leer (ficheros, tablas…) | la aplicación |
| **Prompts** | plantillas reutilizables | el usuario |

En este lab: (1) escribimos un servidor, (2) lo probamos desde el notebook, (3) **se lo conectamos a un LLM** para que lo use solo
y (4) en local, lo exploramos con el **MCP Inspector**. Los pasos 1-3 funcionan también en Colab.

### 🧭 Antes de empezar

- **Cómo se usa:** ejecuta las celdas en orden, de arriba abajo, con `Mayús + Enter`. Si una celda da error, lee la última línea: casi siempre dice qué falta.
- **Si algo se queda raro:** menú *Entorno de ejecución → Reiniciar y ejecutar todo*.
- **Tu API key:** en Colab, guárdala como secreto `OPENAI_API_KEY` (icono de la llave, a la izquierda). Nunca la pegues dentro de una celda.
- **Los pasos 1-4 funcionan en Colab;** el 5 y el 6 (servidor HTTP e Inspector) son en tu ordenador.

**Vocabulario de este notebook**

| Término | Qué es |
|---|---|
| **MCP** | el estándar para conectar un agente con herramientas y datos externos. |
| **Servidor MCP** | el programa que ofrece las herramientas. Lo escribimos nosotros en este lab. |
| **Cliente / host** | la aplicación que se conecta al servidor y usa sus herramientas (un agente, VS Code…). |
| **stdio / HTTP** | las dos formas de conexión: en tu ordenador o por internet. |
| **MCP Inspector** | una herramienta para probar el servidor a mano, sin agente. |


## Paso 1 — Instalar

In [ ]:
!pip install -q --upgrade "mcp>=2.3" openai

> Usamos **`mcp` 2.x**. Si ves tutoriales con `from mcp.server.fastmcp import FastMCP`, son de la versión 1:
> en la 2 la clase se llama **`MCPServer`** (`from mcp.server.mcpserver import MCPServer`).

## Paso 2 — El servidor
Cada tool es una función Python normal con el decorador `@mcp.tool()`. **El nombre, los tipos y el docstring son lo que ve el
modelo** para decidir cuándo usarla: escríbelos como si fueran la documentación para un compañero.

In [ ]:
%%writefile mcp_server_demo.py
"""Servidor MCP de ejemplo: KPIs de una empresa ficticia (Bicis Nube)."""
from mcp.server.mcpserver import MCPServer

mcp = MCPServer("KPIs Bicis Nube")

KPIS = {
    "ventas": {"ingresos_eur": 142000, "clientes_nuevos": 38, "ticket_medio_eur": 3740},
    "marketing": {"leads": 520, "conversion_pct": 7.3, "coste_lead_eur": 42},
    "operaciones": {"viajes": 18400, "averias": 61, "satisfaccion": 4.6},
}


@mcp.tool()
def consultar_kpi(departamento: str, mes: str) -> dict:
    """Devuelve los KPIs de un departamento en un mes.

    Args:
        departamento: ventas, marketing u operaciones.
        mes: mes en formato AAAA-MM, por ejemplo 2026-10.
    """
    dept = departamento.lower()
    if dept not in KPIS:
        return {"error": f"Departamento '{departamento}' no existe. Opciones: {', '.join(KPIS)}."}
    return {"departamento": dept, "mes": mes, **KPIS[dept]}


@mcp.tool()
def convertir_divisa(importe: float, origen: str, destino: str) -> dict:
    """Convierte un importe entre EUR, USD, GBP y JPY con tasas de referencia."""
    tasas = {"EUR": 1.0, "USD": 1.08, "GBP": 0.86, "JPY": 160.0}
    o, d = origen.upper(), destino.upper()
    if o not in tasas or d not in tasas:
        return {"error": f"Divisa no soportada. Opciones: {list(tasas)}"}
    return {"resultado": round(importe / tasas[o] * tasas[d], 2), "divisa": d}


@mcp.tool()
def calcular_roi(inversion_eur: float, beneficio_eur: float) -> dict:
    """Calcula el ROI (%) de una inversión a partir de lo invertido y lo recuperado."""
    if inversion_eur <= 0:
        return {"error": "La inversión debe ser mayor que 0."}
    return {"roi_pct": round((beneficio_eur - inversion_eur) / inversion_eur * 100, 2)}


@mcp.resource("kpis://departamentos")
def departamentos() -> str:
    """Lista de departamentos disponibles."""
    return ", ".join(KPIS)


@mcp.prompt()
def informe_mensual(departamento: str) -> str:
    """Plantilla para pedir un informe mensual de un departamento."""
    return (f"Consulta los KPIs de {departamento} del último mes y escribe un informe de 5 líneas "
            "para dirección: qué va bien, qué va mal y una recomendación.")


if __name__ == "__main__":
    # En local: python mcp_server_demo.py  -> http://localhost:8000/mcp
    # Sin estado y con respuestas JSON (como pide la especificación de 2026): así funciona con más clientes
    mcp.run(transport="streamable-http", port=8000, stateless_http=True, json_response=True)

## Paso 3 — Probarlo desde el notebook
El cliente de `mcp` 2.x (`Client`) se puede conectar a una URL, a un proceso local… o, para probar, **al objeto del servidor
directamente**, sin red. Es lo mismo que hará cualquier cliente: descubrir (`list_*`) y usar (`call_tool`, `read_resource`, `get_prompt`).

In [ ]:
import importlib, json
import mcp_server_demo
importlib.reload(mcp_server_demo)
from mcp import Client

async with Client(mcp_server_demo.mcp) as cliente:
    tools = (await cliente.list_tools()).tools
    print("TOOLS")
    for t in tools:
        print(f"  · {t.name}({', '.join(t.input_schema['properties'])}) — {t.description.splitlines()[0]}")

    r = await cliente.call_tool("calcular_roi", {"inversion_eur": 10000, "beneficio_eur": 13500})
    print("\ncall_tool calcular_roi →", r.content[0].text)

    rec = await cliente.read_resource("kpis://departamentos")
    print("read_resource →", rec.contents[0].text)

    p = await cliente.get_prompt("informe_mensual", {"departamento": "ventas"})
    print("get_prompt →", p.messages[0].content.text)

Fíjate en `input_schema`: MCP genera el **esquema JSON** de cada tool a partir de los tipos de Python. Es exactamente
lo que necesita un LLM para saber cómo llamarla.

## Paso 4 — Dárselo a un LLM
Ahora lo importante: que **un modelo decida solo** qué tools usar. Hacemos el puente a mano (así se ve lo que hacen por dentro
Claude, ChatGPT o el Agents SDK): convertimos las tools MCP al formato de la Responses API y montamos el bucle del agente.

In [ ]:
import os
from getpass import getpass

# La clave NUNCA se escribe en el notebook. Orden: secretos de Colab -> variable de entorno -> teclado
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except Exception:
    if not os.environ.get("OPENAI_API_KEY"):
        os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY: ")
print("Clave cargada:", os.environ["OPENAI_API_KEY"][:7] + "…")

La clave identifica tu cuenta y se factura contra ella. **Nunca** la escribas en el notebook: en Colab, guárdala en *Secretos* (icono de la llave) con el nombre `OPENAI_API_KEY`; en tu ordenador, como variable de entorno. Si no la encuentra, te la pide sin mostrarla.

In [ ]:
import logging
from openai import OpenAI

logging.getLogger("httpx").setLevel(logging.WARNING)   # silencia el log de cada petición HTTP
client = OpenAI()
MODELO = "gpt-6-luna"

async def agente_con_mcp(pregunta: str, servidor) -> str:
    async with Client(servidor) as mcp_cliente:
        # 1) Las tools MCP se convierten en tools de función de OpenAI (nombre + descripción + esquema)
        tools = [{"type": "function", "name": t.name, "description": t.description, "parameters": t.input_schema}
                 for t in (await mcp_cliente.list_tools()).tools]
        entrada = [{"role": "user", "content": pregunta}]
        r = client.responses.create(model=MODELO, reasoning={"effort": "low"}, tools=tools, input=entrada)
        # 2) Bucle: mientras el modelo pida tools, las ejecutamos en el servidor MCP y le devolvemos el resultado
        while llamadas := [o for o in r.output if o.type == "function_call"]:
            salidas = []
            for ll in llamadas:
                res = await mcp_cliente.call_tool(ll.name, json.loads(ll.arguments))
                print(f"  🔧 {ll.name}({ll.arguments}) → {res.content[0].text.replace(chr(10), ' ')[:90]}")
                salidas.append({"type": "function_call_output", "call_id": ll.call_id, "output": res.content[0].text})
            r = client.responses.create(model=MODELO, reasoning={"effort": "low"}, tools=tools,
                                        previous_response_id=r.id, input=salidas)
        return r.output_text

print(await agente_con_mcp(
    "¿Qué tal fue marketing en octubre de 2026? Si invertimos lo que costaron los leads y las ventas fueron "
    "142.000 €, ¿qué ROI sale? Dame también los ingresos en dólares.", mcp_server_demo.mcp))

El modelo **encadena varias tools** por su cuenta: consulta KPIs, calcula y convierte. Nosotros no le hemos dicho en qué orden.
Este mismo servidor, sin cambiar una línea, lo puede usar Claude Desktop, VS Code o el Agents SDK (lo veremos en la clase 4).

## Paso 5 — En local: servidor HTTP + MCP Inspector
> Este paso necesita tu ordenador (en Colab no hay `localhost` accesible desde el navegador).

**Terminal 1** — arranca el servidor:
```bash
python mcp_server_demo.py
```
**Terminal 2** — abre el Inspector (un «Postman para MCP», necesita Node.js 20+):
```bash
npx @modelcontextprotocol/inspector
```
En el navegador: *Transport Type* → **Streamable HTTP**, URL → `http://localhost:8000/mcp`, **Connect**.
Prueba las tools, los resources y los prompts, y mira en el historial el JSON-RPC que viaja.

Con el servidor arrancado, el mismo `Client` se conecta por HTTP:

In [ ]:
try:
    async with Client("http://localhost:8000/mcp") as remoto:
        print("Conectado por HTTP. Tools:", [t.name for t in (await remoto.list_tools()).tools])
except Exception as e:
    print("No hay servidor en localhost:8000 (arráncalo con: python mcp_server_demo.py).", type(e).__name__)

## Paso 6 — Conectarlo a tu asistente
Para que Claude Desktop, Cursor o VS Code lo usen, se añade al fichero de configuración del cliente. En local (el cliente lanza el proceso):
```json
{
  "mcpServers": {
    "kpis-bicis": {
      "command": "python",
      "args": ["/ruta/a/mcp_server_demo.py"]
    }
  }
}
```
Si el servidor estuviera publicado en internet, bastaría con `"type": "http", "url": "https://.../mcp"`.
Desde el terminal, con **Antigravity CLI**: `agy mcp add -t http kpis http://localhost:8000/mcp` (lo vemos en la parte de agentes de programación).

> **Seguridad:** un servidor MCP de terceros es código que lee tus datos y actúa en tu nombre. Instala solo los que conozcas,
> revisa sus permisos y no metas tokens en ficheros que subes a Git (en la clase 4 vemos *prompt injection*).

## Resumen

| Concepto | Qué es |
|---|---|
| `MCPServer` + `@mcp.tool()` | crea un servidor y expone funciones como tools |
| `@mcp.resource` / `@mcp.prompt` | datos de lectura y plantillas |
| `Client(...)` | cliente MCP: URL, proceso local o el objeto del servidor |
| `list_tools` → `call_tool` | descubrir y ejecutar |
| Puente a un LLM | tools MCP → tools de función → bucle del agente |
| MCP Inspector | interfaz para explorar y depurar servidores |

**Para el reto:** el nivel avanzado del reto de vibe coding te pide tu propio servidor MCP. Este notebook es tu plantilla.